# Mixture of Agents — Parallel Proposers + Synthesizer with LangGraph Send()

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/86-mixture-of-agents/mixture_of_agents_workbook.ipynb)

Wang et al. 2024 introduced the **Mixture of Agents (MoA)** pattern: instead of asking one LLM, ask N diverse "proposer" models in parallel and synthesize their answers. Diverse perspectives reduce individual model biases and increase coverage.

---

### Workshop Roadmap

| # | Section |
|---|----------|
| 1 | **MoA Concept** — Wang et al. 2024, ensemble reasoning |
| 2 | **Send() Fan-out** — LangGraph parallel dispatch |
| 3 | **Proposer Nodes** — persona-based parallel answerers |
| 4 | **Aggregator** — synthesis LLM |
| 5 | **Full Run** — end-to-end MoA workflow |
| ★ | **Exercises** — add personas, tune temperature |

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
        'langgraph', 'langchain-openai', 'python-dotenv'])
    print('Colab install complete.')
else:
    print('Local environment — skipping install.')

In [ ]:
import os

# Colab stores secrets in userdata; local dev uses a .env file
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get('OPENAI_API_KEY', '')
print('API key loaded.' if key else 'WARNING: OPENAI_API_KEY not set.')

## Part 1 — The MoA Concept

Wang et al. 2024 showed that using an ensemble of LLM "proposers" and a synthesizer significantly outperforms any single model. The key insight: **language models can judge and combine answers better than they can generate them alone.**

```
Question
    ↓ (fan-out via Send())
[Pragmatist]  [Theorist]  [Educator]   ← run in parallel
    ↓              ↓           ↓
[Aggregator LLM — synthesizes best insights]
    ↓
Final Answer
```

**LangGraph's `Send()`** injects a new state snapshot directly into a node, enabling true parallel dispatch without shared mutable state. Each `Send(node_name, state_dict)` is an independent invocation.

**`Annotated[list[str], operator.add]`**: the reducer merges results from all parallel branches by appending lists together. Each proposer returns `{"proposals": ["its answer"]}` and the reducer concatenates all three lists into one.

In [ ]:
import operator
from typing import Annotated, TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send

PERSONAS = [
    {
        'name': 'Pragmatist',
        'instruction': 'You are a pragmatic engineer. Answer concisely, focusing on practical trade-offs.',
    },
    {
        'name': 'Theorist',
        'instruction': 'You are a computer scientist. Answer with precision and formal definitions.',
    },
    {
        'name': 'Educator',
        'instruction': 'You are a patient teacher. Use simple language and analogies for beginners.',
    },
]

class MoAState(TypedDict):
    question: str
    # operator.add merges lists from parallel branches — each proposer appends one item
    proposals: Annotated[list[str], operator.add]
    final_answer: str

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.7)
print(f'State defined. {len(PERSONAS)} personas: {[p["name"] for p in PERSONAS]}')
print('proposals uses operator.add reducer — parallel branches append without overwriting.')

## Part 2 — The Send() Dispatch Pattern

A **conditional edge** can return a **list of `Send` objects** instead of a single node name. LangGraph executes all of them concurrently — each is an independent invocation of the target node.

```python
# What dispatch() returns — a list of Send objects:
# [
#   Send("proposer", {"question": "...", "persona": PERSONAS[0]}),
#   Send("proposer", {"question": "...", "persona": PERSONAS[1]}),
#   Send("proposer", {"question": "...", "persona": PERSONAS[2]}),
# ]
#
# Each Send creates an independent invocation of the "proposer" node.
# All three run concurrently — no shared state between them.
# Results merge via operator.add on the "proposals" field.
```

Key rules:
- Each `Send` receives a **snapshot** of state — no cross-contamination
- The target node name (`"proposer"`) must be registered in the graph
- `add_conditional_edges(START, dispatch, ["proposer"])` tells LangGraph the possible destinations

In [ ]:
def dispatch(state: MoAState) -> list:
    '''Fan out: one Send per persona — all proposers run in parallel.'''
    return [
        Send('proposer', {'question': state['question'], 'persona': persona})
        for persona in PERSONAS
    ]

def proposer(state: dict) -> dict:
    '''Generate one answer from a single persona.

    Returns {"proposals": [text]} — operator.add appends this to the shared list.
    Each proposer runs independently with its own persona instruction.
    '''
    persona = state['persona']
    messages = [
        SystemMessage(content=persona['instruction']),
        HumanMessage(content=state['question']),
    ]
    response = llm.invoke(messages)
    label = f'[{persona["name"]}] {response.content.strip()}'
    # Return a LIST — operator.add will append this to proposals
    return {'proposals': [label]}

print('dispatch() and proposer() defined.')
print('Key: proposer returns {\'proposals\': [text]} not {\'proposals\': text}')
print('operator.add expects list items to concatenate, not strings to overwrite.')

## Part 3 — The Aggregator

Once all proposers finish, `aggregate` receives the **complete `proposals` list** via the `operator.add` reducer. It formats all proposals into one prompt and asks the LLM to synthesize them.

This is the "mixture" step — the aggregator acts as a **meta-reasoner**, extracting the best insights from each diverse perspective and combining them into a single coherent answer.

In [ ]:
AGGREGATOR_INSTRUCTION = (
    'You are a synthesis expert. Combine these perspectives into one '
    'comprehensive, balanced answer that preserves the best insights from each.'
)

def aggregate(state: MoAState) -> dict:
    '''Synthesize all proposals into one final answer.

    By the time this runs, state["proposals"] contains ALL N responses —
    operator.add has merged them from all parallel proposer invocations.
    '''
    proposals_text = '\n\n'.join(state['proposals'])
    messages = [
        SystemMessage(content=AGGREGATOR_INSTRUCTION),
        HumanMessage(content=(
            f'Question: {state["question"]}\n\n'
            f'Proposals from different perspectives:\n{proposals_text}\n\n'
            'Synthesize a final, comprehensive answer:'
        )),
    ]
    response = llm.invoke(messages)
    return {'final_answer': response.content.strip()}

# Build the graph
builder = StateGraph(MoAState)
builder.add_node('proposer',  proposer)
builder.add_node('aggregate', aggregate)

# dispatch returns a list of Send objects — conditional_edges handles this
builder.add_conditional_edges(START, dispatch, ['proposer'])
builder.add_edge('proposer',  'aggregate')
builder.add_edge('aggregate', END)

app = builder.compile()
print('MoA graph compiled.')
print('Flow: START --dispatch--> proposer (x3 parallel) --> aggregate --> END')

## Part 4 — Running MoA

Initialize `proposals=[]` — the `operator.add` reducer will append results from each parallel branch into this list. After `app.invoke()`:

- `result["proposals"]` — list of N individual answers (one per persona)
- `result["final_answer"]` — the synthesized answer from the aggregator

The `[Pragmatist]`, `[Theorist]`, `[Educator]` prefixes in proposals let you verify which persona generated each answer.

In [ ]:
question = 'What is the difference between supervised and unsupervised learning?'

print(f'Question: {question}\n')
result = app.invoke({
    'question': question,
    'proposals': [],   # operator.add will append to this
    'final_answer': '',
})

print('=== Individual Proposals ===')
for i, proposal in enumerate(result['proposals'], 1):
    print(f'\nProposal {i}:')
    print(proposal)

print('\n=== Synthesized Answer ===')
print(result['final_answer'])

## Part 5 — Why MoA Works

The research insight from Wang et al. 2024: **LLMs can judge quality better than they can generate quality.** The aggregator acts as a meta-reasoner that picks the best parts of each proposal.

**Diversity matters**: using the same persona three times gives diminishing returns — you get near-identical proposals that the aggregator can't differentiate. Different personas surface different aspects of an answer:

- **Pragmatist** — practical trade-offs, real-world application
- **Theorist** — formal definitions, edge cases, mathematical precision
- **Educator** — analogies, accessibility, conceptual clarity

You can verify diversity worked by checking the `[Name]` prefix in each proposal — they should cover different angles.

In [ ]:
question2 = 'Why does LangGraph use a graph structure instead of a simple chain?'

result2 = app.invoke({
    'question': question2,
    'proposals': [],
    'final_answer': '',
})

print(f'Question: {question2}\n')
print(f'Proposals received: {len(result2["proposals"])}')
print(f'Personas used: {[p.split("]")[0].lstrip("[") for p in result2["proposals"]]}'
)
print()
print('Final Answer:')
print(result2['final_answer'])

## Exercises

**Exercise 1** — Add a 4th "Devil's Advocate" persona that challenges assumptions and surfaces counterarguments. Rebuild the dispatch function with 4 personas and verify you get 4 proposals.

**Exercise 2** — Change `temperature=0.7` to `temperature=0.0`. Run the same question twice. Do the proposals become more uniform? What does this tell you about diversity in MoA? (Hint: lower temperature = more deterministic = less diverse ensemble.)

In [ ]:
# Exercise 1 — Add a Devil's Advocate persona
PERSONAS_EXTENDED = PERSONAS + [
    {
        'name': "Devil's Advocate",
        'instruction': (
            'You are a critical thinker who challenges assumptions. '
            'Identify weaknesses, edge cases, or counterarguments in the answer.'
        ),
    }
]

# Rebuild dispatch to use the extended persona list
def dispatch_extended(state: MoAState) -> list:
    return [
        Send('proposer', {'question': state['question'], 'persona': p})
        for p in PERSONAS_EXTENDED
    ]

builder2 = StateGraph(MoAState)
builder2.add_node('proposer',  proposer)   # same proposer node
builder2.add_node('aggregate', aggregate)  # same aggregator

builder2.add_conditional_edges(START, dispatch_extended, ['proposer'])
builder2.add_edge('proposer',  'aggregate')
builder2.add_edge('aggregate', END)

app2 = builder2.compile()

result3 = app2.invoke({
    'question': 'Should you always use a vector database for LLM memory?',
    'proposals': [],
    'final_answer': '',
})

print(f'Got {len(result3["proposals"])} proposals (should be 4):')
for p in result3['proposals']:
    preview = p[:80].replace('\n', ' ')
    print(f'  {preview}...')
print()
print('Final Answer:')
print(result3['final_answer'])

---

**Next:** [87 — Vector Memory Agent](../87-vector-memory-agent/vector_memory_agent_workbook.ipynb) — semantic memory with ChromaDB.